# CivicLens — Model Training Notebook

Train two image classifiers using transfer learning (MobileNetV3-Small): **pothole vs non-pothole** and **garbage dump vs non-garbage**. This notebook creates actual model weights and validation metrics when run; no metrics are assumed in advance.

**Before running:** replace `YOUR-USERNAME` in the first code cell with the GitHub username that owns the CivicLens repository. Run in Google Colab with GPU enabled if available (Runtime → Change runtime type → T4 GPU).

**Dataset:** [QR4Change on Mendeley Data](https://data.mendeley.com/datasets/zndzygc3p3/1) describes 2,966 pothole/non-pothole images and 1,971 garbage/non-garbage images, and lists CC BY 4.0. Download the data from its page, review attribution/usage terms, and do not commit the images to GitHub.

**Expected class folders** after extraction:
```text
data/raw/pothole/       # pothole images
data/raw/not_pothole/   # plain road / non-pothole images
data/raw/garbage/       # garbage dump images
data/raw/not_garbage/   # clean / non-garbage images
```

These are two binary classifiers, not a universal civic-issue classifier. Streetlights and open drains remain manual choices. The notebook expects the project repository to be public so Colab can clone its training scripts.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, zipfile, shutil, json

# Replace YOUR-USERNAME with the GitHub account that owns the CivicLens repository.
REPO_URL = 'https://github.com/YOUR-USERNAME/CivicLens.git'
if 'YOUR-USERNAME' in REPO_URL:
    raise ValueError('Edit REPO_URL above and replace YOUR-USERNAME before running this cell.')
!git clone {REPO_URL} /content/civiclens
ROOT = Path('/content/civiclens')
os.chdir(ROOT)
print('Workspace:', ROOT)


## 1. Upload your downloaded dataset archive(s)

Click Run, select the ZIP archive(s) downloaded from the Mendeley dataset page, and wait for upload. Large archives may take a few minutes. If you downloaded separate pothole and garbage archives, upload both. The cell extracts archives into `data/imported/`.

In [ ]:
uploaded = files.upload()
imported = ROOT / 'data' / 'imported'
imported.mkdir(parents=True, exist_ok=True)
for name, payload in uploaded.items():
    archive = ROOT / name
    archive.write_bytes(payload)
    if name.lower().endswith('.zip'):
        dest = imported / Path(name).stem
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(dest)
        print('Extracted:', name, '->', dest)
    else:
        print('Uploaded non-ZIP file:', name, '(please extract/organize it manually)')

print('Imported folders:')
for p in sorted(imported.rglob('*')):
    if p.is_dir() and len(p.relative_to(imported).parts) <= 3:
        print('DIR ', p.relative_to(ROOT))


## 2. Organize images into four class folders

The dataset's folder names may differ from the expected names above. Use the file browser on the left of Colab or the helper cell below to inspect folders. Copy the matching class images into `data/raw/pothole`, `data/raw/not_pothole`, `data/raw/garbage`, and `data/raw/not_garbage`. **Do not combine unrelated images into a positive class.** The dataset description says the data has separate pothole/non-pothole and garbage/non-garbage classes; keep those labels separate.

The next cell creates the destination folders and prints image counts. If any count is zero, finish organizing the dataset before continuing.

In [ ]:
from PIL import Image
EXTS={'.jpg','.jpeg','.png','.webp','.bmp'}
raw = ROOT / 'data' / 'raw'
for cls in ['pothole','not_pothole','garbage','not_garbage']:
    (raw/cls).mkdir(parents=True, exist_ok=True)

def count_images(folder):
    n=0
    for p in folder.rglob('*'):
        if p.is_file() and p.suffix.lower() in EXTS:
            try:
                with Image.open(p) as im: im.verify()
                n+=1
            except Exception: pass
    return n
print('Expected raw class folders:')
for cls in ['pothole','not_pothole','garbage','not_garbage']:
    print(f'{cls:15s} {count_images(raw/cls)} images')
print('Imported archive images:', sum(1 for p in imported.rglob('*') if p.is_file() and p.suffix.lower() in EXTS))


## 3. Install dependencies and create an 80/20 train/validation split

This downloads compatible PyTorch packages and ImageNet pretrained weights when training begins. The validation split is used for model selection, not an independent external test set.

In [ ]:
!pip -q install -r ml/requirements.txt
!python ml/prepare_dataset.py --raw data/raw --out data --copy


## 4. Train both models

Start with 6 epochs. If the runtime is fast and validation metrics are unstable, you can increase to 8–12 epochs, but inspect overfitting. Training freezes the pretrained feature extractor and trains the classification head; this is a practical baseline for a short project deadline.

In [ ]:
!python ml/train.py --data data --out models --epochs 6 --batch-size 32 --lr 0.001


## 5. Review validation metrics

Use these actual outputs in your presentation only after the training cell completes. Do not call validation performance “test accuracy.”

In [ ]:
from pathlib import Path
import json
for task in ['pothole','garbage']:
    path=ROOT/'models'/f'{task}_metrics.json'
    if path.exists():
        m=json.loads(path.read_text())
        print('\n', task.upper())
        print('Validation accuracy:', round(m['validation_accuracy'],4))
        print('Validation macro-F1:', round(m['validation_macro_f1'],4))
        print('Confusion matrix:', m['confusion_matrix'])
        print('Class report:')
        print(json.dumps(m['classification_report'], indent=2))
    else:
        print('Missing metrics:', path)


## 6. Export trained model weights for the CivicLens API

The ZIP contains only trained model weights and metrics, not the dataset. Download it, then place the `.pth` files in the repository's `ml/models/` folder for local inference or deploy them with the API host. Do not publish private or unlicensed images. The `.pth` files are excluded by `.gitignore` by default because binary model files may be large.

In [ ]:
import zipfile
from google.colab import files
export_path = ROOT/'civiclens_models.zip'
with zipfile.ZipFile(export_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in (ROOT/'models').glob('*'):
        if p.is_file(): zf.write(p, arcname=p.name)
print('Created:', export_path, 'size MB:', round(export_path.stat().st_size/1024/1024,2))
files.download(str(export_path))


## 7. Honest reporting checklist

- State dataset source and licence; cite QR4Change / Mendeley Data.
- State the architecture: MobileNetV3-Small with ImageNet transfer learning and a binary classification head for each task.
- Report the validation split, class counts, validation accuracy, macro-F1, and confusion matrices from this run.
- Mention limitations: dataset bias, domain shift, class imbalance, uncertain predictions, and only two supported issue types.
- The model does not detect streetlights/open drains and does not submit government complaints. Human verification is required.
